# 1.2 Population and labels

**What this notebook is for.** Understanding the label side of the problem before
modelling it. Three questions:

1. How much confirmed fraud is there, and how is it spread across fraud types?
2. What does the fraud taxonomy actually collapse those types into, and what
   falls through to `UNMAPPED_CATEGORY`?
3. How much contamination is there — fraud customers sitting in the non-fraud
   training population, which is the one thing that reliably breaks an
   unsupervised detector.

**Why the taxonomy matters.** The fraud feeds use free-text fraud-type labels
that differ per source and drift over time. `trust_score_05.ml.taxonomy` maps
them onto a small fixed set of `CATEGORIES`. Anything the mapping does not
recognise becomes `UNMAPPED_CATEGORY` rather than being dropped, so growth in the
unmapped bucket is the signal that the mapping needs updating. Check it every
time a new feed arrives.

**Why contamination matters more.** The model learns "normal" from the training
non-fraud population. Every fraud customer left in there teaches it that fraud is
normal, and the harm is worse than the row count suggests, because fraud rows are
exactly the ones sitting where the decision boundary needs to be.
`remove_fraud_customers` in `common/splitting.py` is the fix, and it is applied
by customer id rather than case id on purpose: one fraud case implicates the
whole customer.

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# The notebooks live two levels below the repo root. Put the root on sys.path so
# `import trust_score_05` resolves without an editable install, which is what
# happens on EMR and SageMaker where the package arrives as a zip.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("repo root:", REPO_ROOT)

In [ ]:
from trust_score_05.ml.config import load_ml_config

ENV = "dev"
cfg = load_ml_config(env=ENV)
print("base path:", cfg.base_path)

In [ ]:
from trust_score_05.ml.jobs.base import build_spark_session

spark = build_spark_session("notebook")
spark.sparkContext.setLogLevel("WARN")
spark

## The fraud population

`load_fraud_frame` reads every discovered fraud path, unions them with
`add_missing_and_select` so a feed missing a column does not break the union, and
adds `SOURCE_PATH_COLUMN` so each row remembers where it came from. That last
column is what makes the per-source breakdown below possible, and it is the
reason the multi-path parquet read had to be fixed to stop losing partition
columns — that defect is written up in the ML findings document.

`features=None` means "read every column"; pass a feature list once you know
which ones you want, because the fraud frame is wide.

In [ ]:
from trust_score_05.ml.datasets import (
    FRAUD_META_COLUMNS,
    FRAUD_TIMESTAMP_COLUMN,
    LABEL_COLUMN,
    SOURCE_PATH_COLUMN,
    load_fraud_frame,
)

fraud = load_fraud_frame(spark, cfg, features=None)
fraud.cache()

print("fraud rows      :", fraud.count())
print("label column    :", LABEL_COLUMN)
print("timestamp column:", FRAUD_TIMESTAMP_COLUMN)
print("meta columns    :", FRAUD_META_COLUMNS)

## Cases versus rows

A fraud *case* can appear as several rows — one per feed that reported it, or one
per snapshot it survived into. `deduplicate_fraud` collapses to one row per
`CASE_KEY_COLUMN`. Metrics computed on the un-deduplicated frame overcount, and
they overcount unevenly, because the cases reported by several feeds are the
well-known ones.

In [ ]:
from trust_score_05.common.splitting import CASE_KEY_COLUMN, CUSTOMER_KEY_COLUMN

rows = fraud.count()
cases = fraud.select(CASE_KEY_COLUMN).distinct().count()
customers = fraud.select(CUSTOMER_KEY_COLUMN).distinct().count()

print(f"rows      : {rows}")
print(f"cases     : {cases}  ({rows / max(cases, 1):.2f} rows per case)")
print(f"customers : {customers}  ({cases / max(customers, 1):.2f} cases per customer)")

## Where the fraud comes from

In [ ]:
import pyspark.sql.functions as F

from trust_score_05.ml.datasets import fraud_source_from_path

by_source = (
    fraud.groupBy(SOURCE_PATH_COLUMN)
    .count()
    .orderBy(F.col("count").desc())
    .toPandas()
)
by_source["source"] = by_source[SOURCE_PATH_COLUMN].map(fraud_source_from_path)
by_source[["source", "count"]]

## The taxonomy, and what falls out of it

`normalize_fraud_type` lowercases and strips; `category_for` maps the normalised
type onto a category; `resolve_fraud_source` maps a path onto one of
`FRAUD_SOURCES`. The two mapping tables — `NBC_FRAUD_TYPE_TO_CATEGORY` and
`PHASE1_FRAUD_TYPE_TO_CATEGORY` — are separate because the two feeds use
different vocabularies for the same thing.

**Read the unmapped list.** Every entry is either a new fraud type that needs a
mapping decision, or a typo upstream. Neither resolves itself.

In [ ]:
from trust_score_05.ml.taxonomy import (
    CATEGORY_ORDER,
    UNMAPPED_CATEGORY,
    category_for,
    normalize_fraud_type,
)

FRAUD_TYPE_COL = "fraud_type"  # adjust if the feed names it differently

types = (
    fraud.groupBy(FRAUD_TYPE_COL)
    .count()
    .orderBy(F.col("count").desc())
    .toPandas()
)
types["normalized"] = types[FRAUD_TYPE_COL].map(normalize_fraud_type)
types["category"] = types["normalized"].map(category_for)

summary = (
    types.groupby("category")["count"]
    .sum()
    .reindex(CATEGORY_ORDER)
    .dropna()
    .astype(int)
)
print(summary.to_string())
print()

unmapped = types[types["category"] == UNMAPPED_CATEGORY]
print(f"{len(unmapped)} unmapped fraud type(s), {int(unmapped['count'].sum())} rows")
unmapped[[FRAUD_TYPE_COL, "count"]].head(30)

## Contamination

`load_fraud_customer_ids` returns the customer ids to exclude;
`remove_fraud_customers` does the anti-join. The number printed below is the one
to care about: it is how many training rows were teaching the model that fraud is
normal.

In [ ]:
from trust_score_05.common.splitting import remove_fraud_customers
from trust_score_05.ml.datasets import load_training_matrix

fraud_ids = load_fraud_customer_ids(spark, cfg)
print("fraud customer ids:", fraud_ids.count())

training = load_training_matrix(spark, cfg, features=None)
before = training.count()

clean = remove_fraud_customers(training, fraud_ids)
after = clean.count()

print(f"training rows before : {before}")
print(f"training rows after  : {after}")
print(f"removed              : {before - after}  ({(before - after) / max(before, 1):.4%})")

## Sampling for a laptop

`stratified_sample` caps the row count while preserving the label ratio. This is
for iterating in a notebook; a real run uses the whole population. The `seed`
argument exists so that "it worked on my sample" is a reproducible claim.

In [ ]:
from trust_score_05.common.splitting import stratified_sample

sample = stratified_sample(clean, max_rows=50_000, seed=cfg.seed)
print("sampled rows:", sample.count())

pdf = sample.limit(2000).toPandas()
print("pandas shape:", pdf.shape)
pdf.head()

## What next

- Unmapped types growing? Update the mapping tables in `ml/taxonomy.py` and add a
  test — `tests/ml/test_taxonomy.py` is where the existing ones live.
- Contamination above a fraction of a percent? Ask upstream why fraud customers
  are in the non-fraud extract; the fix belongs there, not here.
- Otherwise: `2.0-features/2.1-build-feature-matrix.ipynb` if you need to rebuild
  features, or straight to `2.2-feature-selection.ipynb` if the matrix is current.